In [ ]:
# —— 形成月快照（S4）：每只股票每个形成月保留最后一个交易日的一行 ——
# 市值列沿用 src_jup/features_3.ipynb 的 get_market_feature。
# (股票, 形成月) 内最后一行即当月最后交易日：先按 [股票, 交易日] 排序，
# 再用 drop_duplicates(keep='last') 定位月末，这是既定写法
# （详见 learning/基础函数使用手册/drop_duplicates详解.md）。
df = get_market_feature(df)
df[form] = df[date].dt.to_period('M')
df = df.sort_values([stock_code, date])
snap = df.drop_duplicates([stock_code, form], keep='last')
snap = snap[[
    stock_code, date, form, day_market_value, eps, pb, roe,
    public_date, end_date, close_price,
]].copy()
snap = snap.rename(columns={day_market_value: monthly_market_value})
snap['formation_end'] = snap[form].dt.to_timestamp(how='end').dt.normalize()
snap[ym] = snap[form] + 1



In [ ]:
# —— 论文流动性条件：过去 12 个自然月交易记录数与形成月交易记录数 ——
# “过去 12 个月”要求窗口在自然月上连续：先把 (股票, 形成月) 铺到完整
# 月历上（没有交易记录的月记 0 天），再按股票滚动 12 行求和
# （一行 = 一个自然月）；滚动写法与 features_3 的 _roll_by_stock 同构
# （按股票分组滚动），区别是窗口单位由交易日换成自然月。
month_grid = pd.period_range(df[form].min(), df[form].max(), freq='M')
count_table = df.groupby([stock_code, form]).size()
count_table = count_table.reset_index(name='trading_days_month')

# 按股票逐个补齐月份，再计算连续12个月的交易日数。
count_table_list = []
for current_stock_code, stock_count in count_table.groupby(stock_code):
    stock_count = stock_count.set_index(form)
    stock_count = stock_count.reindex(month_grid, fill_value=0)
    stock_count.index.name = form
    stock_count['trading_days_year'] = stock_count[
        'trading_days_month'
    ].rolling(12, min_periods=12).sum()
    stock_count[stock_code] = current_stock_code
    count_table_list.append(stock_count.reset_index())

count_table = pd.concat(count_table_list, ignore_index=True)
snap = snap.merge(
    count_table,
    on=[stock_code, form],
    how='left',
    validate='one_to_one',
)

# 如果某只股票整月停牌，对应月份的交易日数按零计入。
first_trade_date = df.groupby(stock_code)[date].min()
snap['first_observed_date'] = snap[stock_code].map(first_trade_date)
six_months_after_first_trade = (
    snap['first_observed_date'] + pd.DateOffset(months=6)
)
snap['listed_six_months_proxy'] = (
    snap['formation_end'] >= six_months_after_first_trade
)

# 三道条件同时满足才进入形成股票池。
has_one_year_of_trading = snap['trading_days_year'].ge(120)
has_one_month_of_trading = snap['trading_days_month'].ge(15)
snap['eligible'] = (
    snap['listed_six_months_proxy']
    & has_one_year_of_trading
    & has_one_month_of_trading
)

